# 04 · Amplitude time series over a monitored site

SAR's edge for *monitoring* is cadence: the same radar geometry revisits a
site, so a stack of passes is a directly comparable measurement of how bright
(how much backscatter) that place is over time. Collapse each pass to a single
number — the mean backscatter in decibels — and you get an **amplitude time
series**: a plot that says *when* a site got busier or quieter, cloud or dark
notwithstanding.

This is the scalar, whole-scene complement to `umbra timescan` (which keeps the
map and shows *where* activity happened) and to `umbra change` (which compares
two passes). Here we reduce space and keep time.

This notebook needs the `load` extra (xarray + rasterio + numpy):

```bash
pip install "umbra-py[load]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Find a site imaged several times

An Umbra *task* is repeat imaging of one site, so grouping a modest search by
task and taking the busiest task gives us a time series to work with — the same
pattern notebook `03` uses, just kept whole instead of trimmed to two frames.

In [ ]:
from umbra_py import UmbraCatalog

by_task: dict[str, list] = {}
for it in UmbraCatalog().search(
    start="2024-01-01", end="2024-12-31", product_types=["GEC"], limit=24
):
    if "GEC" in it.available_assets and it.task and it.datetime is not None:
        by_task.setdefault(it.task, []).append(it)

series = max(by_task.values(), key=len) if by_task else []
assert len(series) >= 2, "no repeat-imaged GEC task in the sampled window"
print(f"{series[0].task}: {len(series)} dated GEC passes")

## 2 · Keep the series apples-to-apples

`select_change_frames(..., frames=None)` returns the *whole* series, oldest
first — and first drops it to a single polarization, because an HH pass and a VV
pass measure different scattering and their brightness difference would show up
as fake activity in the plot. One polarization in, one clean time series out.

In [ ]:
from umbra_py import select_change_frames

# frames=None keeps every pass (oldest-first), grouped to one polarization.
passes = select_change_frames(series, frames=None)
assert len(passes) >= 2

pol = "/".join(passes[0].polarizations) or "?"
print(f"{len(passes)} passes, polarization {pol}")
for p in passes:
    print("  ", p.datetime.date(), p.id)

## 3 · Reduce each pass to one number

`to_xarray(..., db=True)` streams just a decimated overview of each GEC over HTTP
range requests — no multi-gigabyte download — as a georeferenced `DataArray` of
backscatter in decibels, with the non-positive pixels `log10` can't represent
masked to `NaN`. The NaN-aware mean of that overview is our per-pass scalar. A
small `max_size` keeps every read cheap; the trend is robust to the exact
decimation.

In [ ]:
import numpy as np

from umbra_py import to_xarray

times, mean_db = [], []
for p in passes:
    da = to_xarray(p, asset="GEC", max_size=384, db=True)
    valid = da.values[np.isfinite(da.values)]
    if valid.size == 0:
        continue  # an all-nodata overview — skip this pass
    times.append(p.datetime)
    mean_db.append(float(valid.mean()))

assert len(times) >= 2, "need two passes with valid pixels to form a series"
for t, m in zip(times, mean_db, strict=True):
    print(f"{t.date()}  mean backscatter {m:6.1f} dB")

## 4 · Plot the amplitude time series

Now it's plain arrays — dates against mean decibels. A rising line means the
scene is getting brighter overall (more or stronger reflectors — construction, a
filling yard, vehicles); a dip means it's quieting down. To read *where* on the
scene the change is, reach for `umbra timescan`; to see it in color, `umbra
change`.

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(times, mean_db, marker="o")
    ax.set_ylabel("mean backscatter (dB)")
    ax.set_title(f"{series[0].task} — GEC amplitude time series")
    ax.grid(True, alpha=0.3)
    fig.autofmt_xdate()
    plt.show()
except ImportError:
    print("(install matplotlib to plot the amplitude time series)")

## Where next

- **`umbra timescan`** keeps the map: it summarizes the same series into one
  image where color marks *where* activity happened, not just when.
- **`umbra change`** composites two passes into a green/magenta change image, and
  `umbra change --narrate` (the `ai` extra) reads that composite back in plain
  language, grounded in a per-block dB grid.
- **`umbra watch`** turns this into a *standing* monitor: run the same search on a
  schedule and act only on the passes that are new — the amplitude series above,
  updated as each acquisition lands.

*Contains Umbra open data, licensed under CC BY 4.0.*